In [1]:
import numpy as np
import scanpy as sc
from tqdm import tqdm
import scipy.sparse as sp
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go



In [2]:

def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum 

def find_top_peaks_global_summed(mz_axis, global_spec, window_length=9, polyorder=3, min_prominence =0.01, top_n=100, add_da = 0.0001, da=0.024):

    smoothed = savgol_filter(global_spec, window_length=window_length, polyorder=polyorder)

    # Step 1: Peak detection
    peaks, _ = find_peaks(smoothed, prominence=min_prominence)
    print(f"Initial detected peaks: {len(peaks)}")

    mz_peaks = mz_axis[peaks]
    intensities = smoothed[peaks]

    # Step 2: Sort peaks by intensity descending
    sorted_indices = np.argsort(intensities)[::-1]
    mz_peaks_sorted = mz_peaks[sorted_indices]
    intensities_sorted = intensities[sorted_indices]

    # Step 3: Filter out peaks that are too close (within ±da of a stronger one)
    final_mz = []
    final_intensity = []
    for i, mz in enumerate(mz_peaks_sorted):
        if not any(np.abs(mz - accepted_mz) <= da for accepted_mz in final_mz):
            final_mz.append(mz)
            final_intensity.append(intensities_sorted[i])

    final_mz = np.array(final_mz)
    final_intensity = np.array(final_intensity)

    print(f"Filtered peaks (unique by ±{da} Da): {len(final_mz)}")

    # Step 4: Optional: Top-N filtering after proximity filtering
    if top_n is not None and len(final_mz) > top_n:
        top_indices = np.argsort(final_intensity)[-top_n:][::-1]
        final_mz = final_mz[top_indices]
        final_intensity = final_intensity[top_indices]

    # Step 5: Add intensity of all m/z points in ±da range from global_spectrum
    for i, mz in enumerate(final_mz):
        in_range = np.abs(mz_axis - mz) <= add_da
        # Exclude the exact center peak to avoid doubling
        in_range &= ~np.isclose(mz_axis, mz, atol=1e-8)
        added_intensity = np.sum(smoothed[in_range])
        final_intensity[i] += added_intensity

    # Step 6: Final sort by updated intensity
    final_sorted_indices = np.argsort(final_intensity)[::-1]
    final_mz = final_mz[final_sorted_indices]
    final_intensity = final_intensity[final_sorted_indices]
    return final_mz, final_intensity

def sum_peaks_around_targets_in_each_pixel(adata, selected_mz, flank=2):
    mz_axis = adata.var_names.astype(float).values
    intensity_matrix = adata.X
    result = np.zeros((adata.n_obs, len(selected_mz)))

    for i in tqdm(range(adata.n_obs), desc="Summing Intensities"):
        spectrum = intensity_matrix[i, :].toarray().flatten() if sp.issparse(intensity_matrix) else intensity_matrix[i, :]

        for j, target_mz in enumerate(selected_mz):
            # Find closest m/z index
            idx = np.argmin(np.abs(mz_axis - target_mz))
            start = max(0, idx - flank)
            end = min(len(mz_axis), idx + flank + 1)
            result[i, j] = np.sum(spectrum[start:end])

    # Create a new AnnData with summed intensities
    import anndata
    new_adata = anndata.AnnData(X=result)
    new_adata.obs = adata.obs.copy()
    new_adata.var = pd.DataFrame(index=[f"{mz:.4f}" for mz in selected_mz])

    return new_adata

In [3]:
# File paths
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_9points.h5ad"


# Load and process
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")


# parameters
top_mz_number = 1000
min_peak_to_highest_peak = 0.00001
add_range_da = 0.0001
minimimum_peak_distance_da = 0.024
window_length=9
polyorder=3
neighbour_point_n = 4 #number of neighbour points considered in each side of the peaks


mz_axis, global_spec = create_global_spectrum(adata.X, adata.var_names.astype(float).values)

selected_mz, selected_mz_intensity_globally = find_top_peaks_global_summed(mz_axis, global_spec,
                                                        window_length=window_length,
                                                        polyorder=polyorder, 
                                                        min_prominence =min_peak_to_highest_peak, 
                                                        top_n=top_mz_number, 
                                                        add_da = add_range_da, 
                                                        da=minimimum_peak_distance_da)


adata_top_peaks = sum_peaks_around_targets_in_each_pixel(adata, selected_mz, flank=neighbour_point_n)
adata_top_peaks.write(output_file)
print(f"Reduced AnnData with top {top_mz_number} grouped peaks saved to: {output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
Initial detected peaks: 15497
Filtered peaks (unique by ±0.024 Da): 13577


Summing Intensities: 100%|██████████| 16605/16605 [34:14<00:00,  8.08it/s]


Reduced AnnData with top 1000 grouped peaks saved to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_9points.h5ad


In [4]:
def interactive_stem_spectrum_pixel(mz_axis, intensities, mz_min=None, mz_max=None):

    # Filter by mz range if provided
    if mz_min is not None and mz_max is not None:
        mask = (mz_axis >= mz_min) & (mz_axis <= mz_max)
        mz_axis = mz_axis[mask]
        spectrum = intensities[mask]

    # Create the stem lines (from 0 to intensity)
    fig = go.Figure()

    for mz, intensity in zip(mz_axis, spectrum):
        fig.add_trace(go.Scatter(
            x=[mz, mz],
            y=[0, intensity],
            mode="lines",
            line=dict(color="blue", width=1),
            hoverinfo="skip"
        ))

    # Add the tips of the stems
    fig.add_trace(go.Scatter(
        x=mz_axis,
        y=spectrum,
        mode='none',
        hovertemplate='m/z: %{x:.4f}<br>Intensity: %{y:.2f}',
        name='Peaks'
    ))

    fig.update_layout(
        title=f"Interactive Stem Plot",
        xaxis_title="m/z",
        yaxis_title="Intensity",
        template="plotly_white",
        height=400,
        hovermode="closest"
    )

    fig.show()

In [5]:
interactive_stem_spectrum_pixel(selected_mz, selected_mz_intensity_globally, mz_min=0, mz_max=2000)

In [6]:
adata = sc.read_h5ad(output_file)

In [7]:
def plot_mz_across_sample_custom(adata, target_mz, tol=0.1):
    """
    Plots the intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Extract spatial coordinates (assumed to be in adata.obs["x"] and adata.obs["y"])
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    import pandas as pd
    df = pd.DataFrame({"x": x, "y": y, "intensity": intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale = [
    [0.0, "#000000"],  # black
    [0.10, "#000080"],  # navy
    [0.15, "#0000FF"],  # blue
    [0.30, "#8000FF"],  # purple-ish
    [0.45, "#FF0000"],  # red
    [0.60, "#FF8000"],  # orange
    [0.75, "#FFFF00"],  # yellow
    [1.0, "#FFFFFF"]   # white
],
        title=f"Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    fig.show()

In [8]:
plot_mz_across_sample_custom(adata, target_mz=273.0397, tol=0.0001)


In [9]:
adata.var_names 

Index(['273.0397', '137.0251', '274.0435', '331.0297', '369.3523', '257.0478',
       '195.0905', '637.3020', '313.0337', '798.5391',
       ...
       '702.5467', '310.0742', '553.9998', '1271.0072', '252.9273', '386.2827',
       '884.0076', '621.6046', '703.5655', '564.2518'],
      dtype='object', length=1000)

In [10]:
plot_mz_across_sample_custom(adata, target_mz=703.5655, tol=0.0001)


In [11]:
plot_mz_across_sample_custom(adata, target_mz=798.5422, tol=0.01)